In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader

import torch.nn as nn
import torch.optim as optim
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights

import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader

# Define transforms - COMPLETE THE MISSING PARTS
transform = transforms.Compose([
    # TODO: Resize to 28x28
    transforms.Resize((28, 28)),
    transforms.Grayscale(3),  # Convert grayscale to RGB (Don't Touch!!)
    # TODO: Convert to Tensor
    transforms.ToTensor(),
    # TODO: Normalize with ImageNet mean=[0.485, 0.456, 0.406] and std=[0.229, 0.224, 0.225]
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Load EMNIST letters dataset (given)
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# Note: EMNIST letters has labels 1-26 (A-Z), so we have 26 classes
num_classes = 26

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

In [ ]:
# Letter mapping (labels are 1-26 for A-Z)
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

# Create DataLoaders and display samples
# Write your code here

BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

def denormalize(img_t):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img = img_t.cpu() * std + mean
    return img.clamp(0, 1)

# to display some samples
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 6, figsize=(12, 4))
for i in range(12):
    x, y = train_dataset[i]
    y0 = int(y) - 1  # 1-26 -> 0-25
    ax = axes[i // 6, i % 6]
    ax.imshow(denormalize(x).permute(1, 2, 0))
    ax.set_title(letters[y0])
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_v2_s

# Write your code here
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

weights = EfficientNet_V2_S_Weights.IMAGENET1K_V1
model = efficientnet_v2_s(weights=weights)

# Freeze the backbone
for param in model.features.parameters():
    param.requires_grad = False

# Replace classifier head
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, num_classes)

model = model.to(device)


In [ ]:
# Write your code here
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader, desc="Train", leave=False):
        images = images.to(device)
        # TODO: Complete the training step
        # 1. Forward pass
        # 2. Compute loss
        # 3. Zero gradients
        # 4. Backward pass
        # 5. Update weights

        # YOUR CODE HERE
        labels = (labels - 1).to(device).long()

        optimizer.zero_grad()

        logits = model(images)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += images.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader, desc="Val", leave=False):
        images = images.to(device)
        labels = (labels - 1).to(device).long()  # 1-26 -> 0-25

        logits = model(images)
        loss = criterion(logits, labels)

        total_loss += loss.item() * images.size(0)
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += images.size(0)

    return total_loss / total, correct / total


In [ ]:
# Write your code here

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)

num_epochs = 5

train_losses = []
val_losses = []
train_accs = []
val_accs = []

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}, "
          f"Train Acc = {train_acc:.4f}, Val Acc = {val_acc:.4f}")

# Plot the training and validation losses
plt.figure(figsize=(10, 4))
plt.plot(range(1, num_epochs + 1), train_losses, marker='o', label="Train Loss")
plt.plot(range(1, num_epochs + 1), val_losses, marker='o', label="Val Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()
plt.grid(True)
plt.show()

# Plot the training and validation Accuracy
plt.figure(figsize=(10, 4))
plt.plot(range(1, num_epochs + 1), train_accs, marker='o', label="Train Acc")
plt.plot(range(1, num_epochs + 1), val_accs, marker='o', label="Val Acc")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.title("Accuracy Curve")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Write your code here
@torch.no_grad()
def validate_tta(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader, desc="Val TTA", leave=False):
        images = images.to(device)
        labels = (labels - 1).to(device).long()  # 1-26 -> 0-25

        # original
        logits0 = model(images)

        # horizontally flipped
        h_flipped = torch.flip(images, dims=[3])
        logits1 = model(h_flipped)

        # vertically flipped
        v_flipped = torch.flip(images, dims=[2])
        logits2 = model(v_flipped)

        # average predictions
        logits = (logits0 + logits1 + logits2) / 3.0

        loss = criterion(logits, labels)
        total_loss += loss.item() * images.size(0)

        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += images.size(0)

    return total_loss / total, correct / total

tta_loss, tta_acc = validate_tta(model, test_loader, criterion, device)
print(f"TTA: Val Loss = {tta_loss:.4f}, Val Acc = {tta_acc:.4f}")